# Synthetic Dummy Data Generator (UMKM)
Notebook ini membuat data sintetis yang meniru pola data bisnis riil UMKM untuk analisis operasional, NLP, dan klasifikasi bisnis.

Fitur utama yang dihasilkan:

- Monthly_Revenue (IDR)
- Net_Profit_Margin (%)
- Burn_Rate_Ratio
- Transaction_Count
- Avg_Historical_Rating
- Review_Text
- Review_Volatility
- Business_Tenure_Months
- Repeat_Order_Rate (%)
- Digital_Adoption_Score
- Peak_Hour_Latency
- Location_Competitiveness
- Sentiment_Score (-1.0 s/d 1.0)
- Class (Elite, Growth, Struggling, Critical)

Karakteristik realisme yang dimodelkan:

- Korelasi antar fitur operasional (contoh: adopsi digital cenderung meningkatkan retensi dan rating)
- Trade-off bisnis (kompetisi tinggi dan latency tinggi menekan profitabilitas)
- Distribusi finansial tidak simetris (AOV lognormal)
- Noise terkontrol agar data tidak terlalu "rapi"
- Review teks konsisten dengan sinyal kualitas (rating, volatility, latency)
- Sentiment score diekstrak dari `Review_Text`

Catatan target:

- Variabel target adalah `Class` (kolom paling kanan)
- Klasifikasi menggunakan threshold berbasis persentil agar distribusi kelas lebih seimbang
- Urutan kelas: `Elite`, `Growth`, `Struggling`, `Critical`

In [29]:
import random
from typing import List

import numpy as np
import pandas as pd
from faker import Faker

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

# Config
N_SAMPLES = 150000
OUTPUT_CSV = "synthetic_umkm_data.csv"

# Review templates aligned with sentiment
POSITIVE_REVIEWS = [
    "Pelayanan cepat dan ramah, pesanan selalu tepat.",
    "Kualitas produk konsisten, harga masih masuk akal.",
    "Aplikasi pemesanan mudah dipakai dan responsif.",
    "Pengiriman cepat, admin komunikatif.",
    "Selalu repeat order karena kualitasnya terjaga.",
    "Transaksi digital lancar, proses checkout tidak ribet.",
]

NEUTRAL_REVIEWS = [
    "Produk cukup baik, kadang waktu tunggu agak lama.",
    "Pelayanan standar, masih bisa ditingkatkan.",
    "Harga dan kualitas seimbang, pengalaman biasa saja.",
    "Kadang stok kosong saat jam ramai.",
    "Secara umum oke, hanya respon chat kadang lambat.",
]

NEGATIVE_REVIEWS = [
    "Pesanan sering terlambat saat jam sibuk.",
    "Kualitas tidak konsisten, kadang bagus kadang kurang.",
    "Respons admin lambat dan informasi kurang jelas.",
    "Proses pembayaran sering bermasalah.",
    "Harga naik tapi layanan tidak membaik.",
    "Sudah beberapa kali order, pengalaman makin menurun.",
]


def clamp(x: np.ndarray, low: float, high: float) -> np.ndarray:
    return np.clip(x, low, high)


def pick_review(rating: float, volatility: float, latency: str) -> str:
    """Generate review text coherent with quality signal."""
    base_pool: List[str]

    if rating >= 4.2 and volatility < 0.45 and latency == "Low":
        base_pool = POSITIVE_REVIEWS
    elif rating < 3.4 or latency == "High":
        base_pool = NEGATIVE_REVIEWS
    else:
        base_pool = NEUTRAL_REVIEWS

    text = random.choice(base_pool)

    # Add slight random variation so reviews don't look templated
    if random.random() < 0.3:
        text += " " + fake.sentence(nb_words=6)
    return text


def calculate_sentiment_score(review_text: str) -> float:
    """Convert review text into sentiment score in range [-1.0, 1.0]."""
    review_lower = review_text.lower()

    positive_keywords = {
        "cepat": 0.30,
        "ramah": 0.30,
        "mudah": 0.25,
        "responsif": 0.30,
        "lancar": 0.25,
        "komunikatif": 0.25,
        "terjaga": 0.25,
        "konsisten": 0.20,
        "tepat": 0.20,
    }
    negative_keywords = {
        "lambat": -0.30,
        "tidak": -0.20,
        "kurang": -0.25,
        "bermasalah": -0.35,
        "terlambat": -0.35,
        "ribet": -0.30,
        "buruk": -0.40,
        "menurun": -0.30,
    }

    score = 0.0
    for word, weight in positive_keywords.items():
        if word in review_lower:
            score += weight
    for word, weight in negative_keywords.items():
        if word in review_lower:
            score += weight

    return float(clamp(np.array([score]), -1.0, 1.0)[0])


# 1) Business maturity and competitiveness
business_tenure = np.random.randint(3, 180, size=N_SAMPLES)  # months
location_competitiveness = np.random.poisson(lam=8, size=N_SAMPLES) + 1

# 2) Digital adoption (1-10), positively related with tenure (up to a limit)
base_digital = 3.3 + 0.02 * np.sqrt(business_tenure)
noise_digital = np.random.normal(0, 1.8, N_SAMPLES)
digital_adoption = clamp(base_digital + noise_digital, 1, 10)

# 3) Transaction count depends on maturity, digital, and local competition
transaction_lambda = (
    50
    + 0.65 * business_tenure
    + 8.5 * digital_adoption
    - 2.4 * location_competitiveness
    + np.random.normal(0, 18, N_SAMPLES)
)
transaction_lambda = clamp(transaction_lambda, 20, 900)
transaction_count = np.random.poisson(transaction_lambda).astype(int)
transaction_count = np.maximum(transaction_count, 5)

# 4) Average order value (AOV) and monthly revenue
aov = np.random.lognormal(mean=np.log(65000), sigma=0.45, size=N_SAMPLES)
aov = clamp(aov, 12000, 450000)

monthly_revenue = transaction_count * aov
seasonality_noise = np.random.normal(1.0, 0.08, N_SAMPLES)
monthly_revenue = monthly_revenue * seasonality_noise
monthly_revenue = clamp(monthly_revenue, 1_500_000, 850_000_000)

# 5) Peak hour latency category influenced by transaction pressure and digital adoption
latency_score = (
    0.0045 * transaction_count
    - 0.28 * digital_adoption
    + 0.09 * location_competitiveness
    + np.random.normal(0, 0.9, N_SAMPLES)
)

peak_hour_latency = np.where(
    latency_score < 0.0,
    "Low",
    np.where(latency_score < 1.3, "Med", "High")
)

# 6) Burn rate ratio (expense/revenue)
latency_penalty = np.select(
    [peak_hour_latency == "Low", peak_hour_latency == "Med", peak_hour_latency == "High"],
    [0.0, 0.10, 0.22],
    default=0.10,
)

burn_rate_ratio = (
    0.80
    + 0.015 * location_competitiveness
    - 0.014 * digital_adoption
    + latency_penalty
    + np.random.normal(0, 0.10, N_SAMPLES)
)
burn_rate_ratio = clamp(burn_rate_ratio, 0.40, 1.55)

# 7) Net profit margin (%), inverse relation with burn rate
net_profit_margin = (
    (1 - burn_rate_ratio) * 100
    + 0.55 * (digital_adoption - 5)
    - 0.18 * np.log1p(location_competitiveness)
    + np.random.normal(0, 3.2, N_SAMPLES)
)
net_profit_margin = clamp(net_profit_margin, -35, 45)

# 8) Repeat order rate (%), boosted by digital adoption and tenure
repeat_order_rate = (
    16
    + 1.9 * digital_adoption
    + 0.03 * business_tenure
    - 0.6 * location_competitiveness
    + np.random.normal(0, 7.0, N_SAMPLES)
)
repeat_order_rate = clamp(repeat_order_rate, 2, 90)

# 9) Review volatility
review_volatility = (
    0.24
    + 0.18 * (peak_hour_latency == "Med").astype(float)
    + 0.34 * (peak_hour_latency == "High").astype(float)
    + 0.06 * (burn_rate_ratio > 1.0).astype(float)
    + np.random.normal(0, 0.09, N_SAMPLES)
)
review_volatility = clamp(review_volatility, 0.06, 1.30)

# 10) Average historical rating (1-5)
avg_historical_rating = (
    3.95
    + 0.08 * digital_adoption
    + 0.016 * net_profit_margin
    - 0.38 * review_volatility
    - 0.12 * (peak_hour_latency == "High").astype(float)
    + np.random.normal(0, 0.26, N_SAMPLES)
)
avg_historical_rating = clamp(avg_historical_rating, 1.0, 5.0)

# 11) Review text generation coherent with rating/volatility/latency
review_text = [
    pick_review(rating=r, volatility=v, latency=l)
    for r, v, l in zip(avg_historical_rating, review_volatility, peak_hour_latency)
]

# 12) Sentiment score derived from review text
sentiment_scores = np.array([calculate_sentiment_score(text) for text in review_text])

# Optional: post-adjustment for severe deficit businesses
deficit_mask = burn_rate_ratio > 1.25
avg_historical_rating[deficit_mask] = np.minimum(
    avg_historical_rating[deficit_mask],
    np.random.uniform(1.5, 3.5, deficit_mask.sum()),
)
repeat_order_rate[deficit_mask] = np.minimum(
    repeat_order_rate[deficit_mask],
    np.random.uniform(3, 30, deficit_mask.sum()),
)

# 13) Target Class with percentile-based thresholds (balanced by design)
target_class = np.full(N_SAMPLES, "Growth", dtype=object)

elite_mask = (
    (net_profit_margin > np.percentile(net_profit_margin, 70))
    & (burn_rate_ratio < np.percentile(burn_rate_ratio, 25))
    & (repeat_order_rate > np.percentile(repeat_order_rate, 70))
    & (avg_historical_rating > np.percentile(avg_historical_rating, 75))
)

critical_mask = (
    (burn_rate_ratio > np.percentile(burn_rate_ratio, 92))
    | ((business_tenure < 7) & (location_competitiveness >= 12))
    | ((net_profit_margin < np.percentile(net_profit_margin, 5)) & (avg_historical_rating < 3.0))
)

struggling_mask = (
    ((net_profit_margin < np.percentile(net_profit_margin, 35)) & (burn_rate_ratio > np.percentile(burn_rate_ratio, 60)))
    | ((peak_hour_latency == "High") & (avg_historical_rating < np.percentile(avg_historical_rating, 40)))
    | ((burn_rate_ratio > np.percentile(burn_rate_ratio, 75)) & (avg_historical_rating < np.percentile(avg_historical_rating, 65)))
)

target_class[elite_mask] = "Elite"
target_class[struggling_mask] = "Struggling"
target_class[critical_mask] = "Critical"

# Final DataFrame (class at the rightmost position)
df = pd.DataFrame(
    {
        "ID": np.arange(1, N_SAMPLES + 1),
        "Monthly_Revenue": np.round(monthly_revenue, 0).astype(int),
        "Net_Profit_Margin (%)": np.round(net_profit_margin, 2),
        "Burn_Rate_Ratio": np.round(burn_rate_ratio, 3),
        "Transaction_Count": transaction_count.astype(int),
        "Avg_Historical_Rating": np.round(avg_historical_rating, 2),
        "Review_Text": review_text,
        "Review_Volatility": np.round(review_volatility, 3),
        "Business_Tenure_Months": business_tenure.astype(int),
        "Repeat_Order_Rate (%)": np.round(repeat_order_rate, 2),
        "Digital_Adoption_Score": np.round(digital_adoption, 2),
        "Peak_Hour_Latency": peak_hour_latency,
        "Location_Competitiveness": location_competitiveness.astype(int),
        "Sentiment_Score": np.round(sentiment_scores, 3),
        "Class": target_class,
    }
)

# Save and preview
df.to_csv(OUTPUT_CSV, index=False)

print(f"Generated {len(df)} rows -> {OUTPUT_CSV}")
print("\nPreview:")
display(df.head(10))

print("\nSummary stats:")
display(df.describe(include="all").transpose())

print("\nClass counts:")
print(df["Class"].value_counts())

Generated 150000 rows -> synthetic_umkm_data.csv

Preview:


,ID,Monthly_Revenue,Net_Profit_Margin (%),Burn_Rate_Ratio,Transaction_Count,Avg_Historical_Rating,Review_Text,Review_Volatility,Business_Tenure_Months,Repeat_Order_Rate (%),Digital_Adoption_Score,Peak_Hour_Latency,Location_Competitiveness,Sentiment_Score,Class
0,1,6680716,22.72,0.811,161,4.75,"Transaksi digital lancar, proses checkout tida...",0.313,105,19.40,4.24,Low,9,-0.25,Growth
1,2,5819101,4.46,0.968,104,4.21,"Harga dan kualitas seimbang, pengalaman biasa ...",0.632,95,14.87,1.27,Med,10,0.00,Growth
2,3,5236404,-10.12,1.047,102,3.51,"Pelayanan standar, masih bisa ditingkatkan.",0.470,17,21.00,3.37,Med,8,0.00,Struggling
3,4,8043552,0.04,0.969,99,4.33,"Transaksi digital lancar, proses checkout tida...",0.206,109,30.62,5.41,Low,13,-0.25,Growth
4,5,6071256,4.22,0.954,115,4.34,Selalu repeat order karena kualitasnya terjaga...,0.232,74,20.87,2.67,Low,7,0.25,Growth
5,6,6683141,29.68,0.727,108,4.54,"Pengiriman cepat, admin komunikatif. Culpa ver...",0.185,23,26.35,5.59,Low,16,0.55,Elite
6,7,14123932,15.28,0.860,167,4.54,"Produk cukup baik, kadang waktu tunggu agak la...",0.434,105,22.15,3.95,Med,6,0.00,Growth
7,8,8483571,8.51,0.862,180,4.83,Selalu repeat order karena kualitasnya terjaga.,0.346,124,23.17,7.59,Low,10,0.25,Growth
8,9,14900709,6.00,0.908,135,4.86,Selalu repeat order karena kualitasnya terjaga...,0.285,77,15.85,6.56,Low,7,0.25,Growth
9,10,9232562,-13.64,1.085,89,4.39,"Transaksi digital lancar, proses checkout tida...",0.182,90,17.30,3.22,Low,9,-0.25,Struggling



Summary stats:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,150000.0,NaN,NaN,NaN,75000.5,43301.414527,1.0,37500.75,75000.5,112500.25,150000.0
Monthly_Revenue,150000.0,NaN,NaN,NaN,8451726.379767,5291163.126671,1500000.0,4745883.75,7245678.5,10830255.25,82067536.0
Net_Profit_Margin (%),150000.0,NaN,NaN,NaN,1.842272,15.002406,-35.0,-8.43,2.16,12.31,45.0
Burn_Rate_Ratio,150000.0,NaN,NaN,NaN,0.969885,0.144039,0.437,0.869,0.966,1.067,1.55
Transaction_Count,150000.0,NaN,NaN,NaN,117.766667,42.618493,9.0,86.0,117.0,149.0,285.0
Avg_Historical_Rating,150000.0,NaN,NaN,NaN,4.061107,0.521698,1.5,3.77,4.1,4.41,5.0
Review_Text,150000,45139,"Produk cukup baik, kadang waktu tunggu agak lama.",11632,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Review_Volatility,150000.0,NaN,NaN,NaN,0.407203,0.166806,0.06,0.278,0.405,0.526,0.99
Business_Tenure_Months,150000.0,NaN,NaN,NaN,91.00684,51.104736,3.0,47.0,91.0,135.0,179.0
Repeat_Order_Rate (%),150000.0,NaN,NaN,NaN,19.980521,8.021928,2.0,14.45,19.95,25.43,54.06



Class counts:
Class
Growth        85678
Struggling    41571
Critical      12561
Elite         10190
Name: count, dtype: int64
